# Symbolic ViZDoom: Translating 3D to LLM Prompts
This notebook demonstrates a pipeline that processes a ViZDoom trajectory and outputs a purely text-based, symbolic representation for every step. 

By intersecting the `labels` buffer (what is actually visible on screen) with the `objects` buffer (the exact 3D coordinates in the map), we can generate a perfect text observation tailored for Language Models.


In [1]:
import os
import json
import numpy as np
import gymnasium as gym
import vizdoom.gymnasium_wrapper

# Load the dataset
DATASET_DIR = os.path.abspath("../data/sub-01_20260909-135403") # Deadly Corridor
manifest_path = os.path.join(DATASET_DIR, "manifest.json")
with open(manifest_path, "r") as f:
    manifest = json.load(f)

game_curriculum = next(p for p in manifest["curriculum"] if p.get("type") == "game")
game_phase = next(p for p in manifest["phases"] if p.get("type") == "game")
data = np.load(os.path.join(DATASET_DIR, game_phase["data_file"]), allow_pickle=True)

env_kwargs = game_curriculum.get("env_kwargs", {})
game_name = game_curriculum["game"]
actions = data["actions"]
seed = int(data["episode_seeds"][0])



## Initialize Environment and Enable Sensors
We need both `labels` (to know what the agent can actually *see*) and `objects` (to know where those things are).


In [2]:
env = gym.make(game_name, render_mode="rgb_array", **env_kwargs)
game = env.unwrapped.game

# Enable buffers
game.close()
game.set_labels_buffer_enabled(True)
game.set_objects_info_enabled(True)
game.init()



## Pipeline Logic
This function takes the raw ViZDoom state at time $t$ and formats it into the exact symbolic text structure requested.


In [3]:
import sys
if os.path.abspath("../") not in sys.path:
    sys.path.append(os.path.abspath("../"))
from fmri_gym.adapters.vizdoom import _get_button_map

available_buttons = [str(b).split(".")[-1] for b in game.get_available_buttons()]

# Call the adapter's own function so if the repo changes, this notebook adapts automatically
button_map = _get_button_map(env)

ACTION_MAPPING = {}
for action_id, row in enumerate(button_map):
    pressed = [available_buttons[j] for j, v in enumerate(row) if v]
    if not pressed:
        ACTION_MAPPING[action_id] = "NO-OP (Do nothing)"
    else:
        ACTION_MAPPING[action_id] = " + ".join(pressed)

print("Exact fmri_gym Action Mapping (Loaded via Adapter):")
for k, v in ACTION_MAPPING.items():
    print(f" ID {k} -> {v}")

def generate_symbolic_prompt(t, state, action, cumulative_reward, gvars):
    # 1. Parse Inventory (Game Variables)
    # In Deadly Corridor, gvars[0] is typically Health.
    health = int(gvars[0]) if len(gvars) > 0 else 100
    inventory_str = f"[Health: {health}]"
    
    # 2. Parse Objects
    # We want to only list objects that are VISIBLE on screen.
    # state.labels contains visible objects. state.objects contains their 3D coordinates.
    objects_str = ""
    
    if state and state.labels and state.objects:
        # Create a fast lookup table for 3D coordinates by object_id
        obj_lookup = {o.id: o for o in state.objects}
        
        for label in state.labels:
            # Skip the player's own weapon/arms appearing on screen
            if label.object_name == "DoomPlayer":
                continue
                
            obj3d = obj_lookup.get(label.object_id)
            if obj3d:
                # Format: - [NAME] at (X, Y)
                x, y = int(obj3d.position_x), int(obj3d.position_y)
                objects_str += f"- {label.object_name} at ({x}, {y})\n"
                
    if not objects_str:
        objects_str = "- None\n"
        
    # 3. Format the Text Block
    prompt = f"[USER  t={t}]\n"
    prompt += "Observation:\n"
    prompt += f"Score: {int(cumulative_reward)}\n"
    prompt += f"Inventory: {inventory_str}\n"
    prompt += "Objects:\n"
    prompt += objects_str
    prompt += "\n"
    
    prompt += f"[ASSISTANT t={t}]\n"
    if isinstance(action, (np.ndarray, list, tuple)) and len(np.shape(action)) > 0 and len(action) > 1:
        # MultiBinary mode: The action is an array of 0s and 1s representing button states
        pressed = [available_buttons[i] for i, v in enumerate(action) if v > 0]
        action_name = " + ".join(pressed) if pressed else "NO-OP (Do nothing)"
    else:
        # Discrete mode: The action is a single integer
        action_name = ACTION_MAPPING.get(int(action), f"UNKNOWN_{action}")
    prompt += f"Action: {action_name}\n"
    
    return prompt



Exact fmri_gym Action Mapping (Loaded via Adapter):
 ID 0 -> NO-OP (Do nothing)
 ID 1 -> TURN_RIGHT
 ID 2 -> TURN_LEFT
 ID 3 -> MOVE_BACKWARD
 ID 4 -> MOVE_FORWARD
 ID 5 -> ATTACK
 ID 6 -> MOVE_RIGHT
 ID 7 -> MOVE_LEFT


## Run the Pipeline
Let's run the environment for a few steps and watch the symbolic prompts stream out!


In [4]:
# Extract episode IDs and seeds to handle deaths/resets
ref_episode_ids = data.get("episode_id", np.zeros(len(actions), dtype=int))
ref_episode_seeds = data["episode_seeds"]

FPS = 35
start_frame = 15 * FPS  # 15 seconds in (Frame 525)
end_frame = min(40 * FPS, len(actions)) # 40 seconds in, or end of recording

print(f"Fast-forwarding to {start_frame / FPS} seconds (Frame {start_frame})...")
print(f"Writing LLM Prompts for frames {start_frame} to {end_frame} into a text file...\n")

# Reset to start
current_ep = -1
cumulative_reward = 0.0

output_file = "symbolic_trajectory_15s_to_40s.txt"
with open(output_file, "w") as f:
    for t in range(1, end_frame + 1):
        ep_id = int(ref_episode_ids[t-1])
        
        # If the recording transitioned to a new episode (e.g. player died and respawned)
        if ep_id != current_ep:
            obs, info = env.reset(seed=int(ref_episode_seeds[ep_id]))
            current_ep = ep_id
            cumulative_reward = 0.0 # Reset score for new episode!
            
        state = game.get_state()
        action = actions[t-1]
        
        if t >= start_frame:
            gvars = state.game_variables if state else []
            prompt = generate_symbolic_prompt(t, state, action, cumulative_reward, gvars)
            f.write(prompt + "\n")
            
            if t == start_frame:
                print("=== PREVIEW OF FIRST SAVED PROMPT ===")
                print(prompt)
                print("=====================================\n")
                
        obs, reward, term, trunc, info = env.step(action)
        cumulative_reward += reward

env.close()
print(f"Pipeline Complete. Saved all prompts to: {output_file}")


Fast-forwarding to 15.0 seconds (Frame 525)...
Writing LLM Prompts for frames 525 to 1035 into a text file...

=== PREVIEW OF FIRST SAVED PROMPT ===
[USER  t=525]
Observation:
Score: 53
Inventory: [Health: 10]
Objects:
- GreenArmor at (1312, 0)
- DeadShotgunGuy at (178, 71)
- Shotgun at (162, 75)

[ASSISTANT t=525]
Action: MOVE_FORWARD


Pipeline Complete. Saved all prompts to: symbolic_trajectory_15s_to_40s.txt


---
# Symbolic Trajectory Hidden States Extraction with Qwen
Instead of raw pixels, we will now feed the **symbolic text trajectory** we just saved in the `.txt` file into a Qwen Language Model. By asking the model to process this text sequence with `output_hidden_states=True`, we can extract the internal neural representations of the game state directly from the Transformer layers!


In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. Load a Qwen Text model (using 0.5B or 1.5B for fast inference and low memory)
model_id = "Qwen/Qwen2.5-0.5B-Instruct"
print(f"Loading {model_id}...")

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)
print("Qwen Model Ready!")



Loading Qwen/Qwen2.5-0.5B-Instruct...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

: 

## Process the Trajectory Step-by-Step
We will load the text file and split it into individual steps. We can then pass each step through Qwen **one at a time** to extract the internal representations for every single frame individually without running into memory issues!


In [ ]:
# Read the saved text file
with open("symbolic_trajectory_15s_to_40s.txt", "r") as f:
    full_trajectory = f.read()

# Split by the [USER token to separate the individual frames/steps
raw_steps = full_trajectory.split("[USER")

# Re-add the [USER token to fix the splitting, and ignore the first empty split
steps = ["[USER" + s for s in raw_steps if s.strip()]

print(f"Loaded {len(steps)} individual steps from the text file.")
print("=== Example of a Single Step ===")
print(steps[0])



## Extract Internal Representations
We will now loop through the steps one by one. For each step, we tokenize it, pass it through the model with `output_hidden_states=True`, and extract the raw activation tensors!


In [ ]:
import time

all_hidden_states = []

print(f"Extracting hidden states one by one following the Passive Replay Protocol...")

# Dynamically construct the [SYSTEM PROMPT] using the actions available in this scenario
action_list = ", ".join(ACTION_MAPPING.values())

system_prompt = "[SYSTEM PROMPT]\n"
system_prompt += "You are playing a 3D action-shooter game. Respond with exactly one action.\n"
system_prompt += f"Available actions: {action_list}.\n"
system_prompt += "Mechanics: Navigate the 3D environment, monitor your Health, and engage enemies. You will be provided with your current score and the (X, Y) map coordinates of all currently visible objects.\n\n"

start_time = time.time()

# Process the first 20 frames as an example
for i, step in enumerate(steps[:20]):
    
    # Split off the action so the model doesn't see the future
    if "Action: " in step:
        user_prompt = step.split("Action: ")[0] + "Action:"
    else:
        user_prompt = step
        
    # Prepend the system prompt to the chopped user step!
    full_prompt = system_prompt + user_prompt
        
    # Independent Forward Pass
    inputs = tokenizer(full_prompt, return_tensors="pt").to(model.device)
    
    with torch.inference_mode():
        outputs = model(
            **inputs,
            output_hidden_states=True,
            return_dict=True
        )
    
    # Extract Hidden State at the Final Real Input Token
    final_layer = outputs.hidden_states[-1]
    final_token_hidden_state = final_layer[0, -1, :]
    
    all_hidden_states.append(final_token_hidden_state.detach().cpu())
    
    if i == 0:
        print(f"\nCaptured first frame!")
        print(f"=== Prompt fed to model ===\n{full_prompt}")
        print(f"=== Extraction ===\nExtracted Hidden State Shape (Last Token): {final_token_hidden_state.shape}")

print(f"\nProcessed {len(all_hidden_states)} independent frames in {round(time.time() - start_time, 2)} seconds.")
print(f"The list `all_hidden_states` now contains the 1D tensor representation for each frame!")
